## 7. Conclusion

### Key Findings from Exploratory Data Analysis

**Dataset Overview:**
- The training dataset contains 1,460 properties with 81 features
- Several features have missing values, requiring imputation strategies
- Mix of numeric and categorical features needs appropriate preprocessing

**Target Variable (SalePrice):**
- SalePrice is right-skewed (skewness > 1), suggesting log transformation would be beneficial
- Prices range from ~$34k to ~$755k with mean around $180k
- Log transformation normalizes the distribution, making it suitable for regression models

**Feature Correlations:**
- **OverallQual**: Strongest predictor (correlation: ~0.79) — higher quality = higher price
- **GrLivArea**: Second strongest (correlation: ~0.71) — more living space = higher price
- **TotalBsmtSF**: Third strongest (correlation: ~0.61) — basement size matters
- **YearBuilt**: Newer homes tend to be more expensive (correlation: ~0.55)

**Outliers:**
- Identified ~2 properties with >4000 sqft living area but <$200k price
- These unusual cases may need special handling or removal during model training
- Consider investigating these properties or flagging them as data quality issues

**Next Steps:**
1. Handle missing values using median (numeric) and mode (categorical) imputation
2. Scale numeric features using StandardScaler to normalize ranges
3. One-hot encode categorical features to convert to numeric format
4. Consider removing or flagging identified outliers
5. Train multiple models (Linear Regression, Random Forest) and compare performance
6. Consider feature engineering to improve model predictions

In [ ]:
# Identify outliers: GrLivArea > 4000 sqft with low SalePrice
# These are unusual data points that might affect model training

outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 200000)]

print(f"\nOutliers Detected: {len(outliers)} properties with GrLivArea > 4000 sqft and SalePrice < $200k")
print("\nOutlier Details:")
print(outliers[['GrLivArea', 'SalePrice', 'OverallQual', 'YearBuilt']].to_string())

# Visualize outliers
plt.figure(figsize=(12, 6))
plt.scatter(train_df['GrLivArea'], train_df['SalePrice'], alpha=0.5, s=50, label='Normal Data')
plt.scatter(outliers['GrLivArea'], outliers['SalePrice'], color='red', s=100, 
           edgecolors='darkred', linewidth=2, label='Outliers', marker='X')
plt.xlabel('Ground Living Area (sqft)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.title('Outlier Detection: GrLivArea vs SalePrice', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Outlier Detection

In [ ]:
# Bar plot: Mean prices by Neighborhood
neighborhood_prices = train_df.groupby('Neighborhood')['SalePrice'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
neighborhood_prices.plot(kind='bar', color='teal', edgecolor='black', alpha=0.7)
plt.xlabel('Neighborhood', fontsize=12)
plt.ylabel('Average Sale Price ($)', fontsize=12)
plt.title('Average Sale Price by Neighborhood', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nTop 5 Most Expensive Neighborhoods:")
print(neighborhood_prices.head())

In [ ]:
# Scatter plot: GrLivArea vs SalePrice
plt.figure(figsize=(12, 6))
plt.scatter(train_df['GrLivArea'], train_df['SalePrice'], alpha=0.5, s=50, edgecolors='k', linewidth=0.5)
plt.xlabel('Ground Living Area (sqft)', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.title('Sale Price vs Ground Living Area', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

# Add trend line
z = np.polyfit(train_df['GrLivArea'], train_df['SalePrice'], 1)
p = np.poly1d(z)
plt.plot(train_df['GrLivArea'].sort_values(), p(train_df['GrLivArea'].sort_values()),
         "r--", linewidth=2, label='Trend Line')
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nCorrelation between GrLivArea and SalePrice: {train_df['GrLivArea'].corr(train_df['SalePrice']):.4f}")

In [ ]:
# Box plot: OverallQual vs SalePrice
plt.figure(figsize=(12, 6))
sns.boxplot(data=train_df, x='OverallQual', y='SalePrice', palette='Set2')
plt.xlabel('Overall Quality', fontsize=12)
plt.ylabel('Sale Price ($)', fontsize=12)
plt.title('Sale Price Distribution by Overall Quality', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5. Key Feature Visualizations

In [ ]:
# Heatmap of top 15 features correlated with SalePrice
top_features = correlations.head(16).index.tolist()  # Top 15 + SalePrice
correlation_matrix = train_df[top_features].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Heatmap: Top 15 Features + SalePrice', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate correlation with SalePrice for numeric features
numeric_features = train_df.select_dtypes(include=[np.number]).columns
correlations = train_df[numeric_features].corr()['SalePrice'].sort_values(ascending=False)

print("\nTop 15 Features Correlated with SalePrice:")
print(correlations.head(16))  # Include SalePrice itself

## 4. Feature Correlations

In [ ]:
# Log-transformed SalePrice distribution
# Log transformation helps normalize right-skewed data
log_saleprice = np.log1p(train_df['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of log-transformed prices
axes[0].hist(log_saleprice, bins=50, edgecolor='black', alpha=0.7, color='lightcoral')
axes[0].set_xlabel('Log(Sale Price)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Log-Transformed Sale Prices', fontweight='bold')
axes[0].grid(alpha=0.3)

# KDE of log-transformed prices
log_saleprice.plot(kind='kde', ax=axes[1], color='darkred', linewidth=2)
axes[1].set_xlabel('Log(Sale Price)')
axes[1].set_ylabel('Density')
axes[1].set_title('KDE of Log-Transformed Sale Prices', fontweight='bold')
axes[1].grid(alpha=0.3)

print(f"Log-transformed SalePrice Skewness: {log_saleprice.skew():.4f}")
print(f"Original SalePrice Skewness: {train_df['SalePrice'].skew():.4f}")

plt.tight_layout()
plt.show()

In [ ]:
# Plot distribution of SalePrice
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
axes[0].hist(train_df['SalePrice'], bins=50, edgecolor='black', alpha=0.7, color='skyblue')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Sale Prices', fontweight='bold')
axes[0].grid(alpha=0.3)

# KDE plot
train_df['SalePrice'].plot(kind='kde', ax=axes[1], color='darkblue', linewidth=2)
axes[1].set_xlabel('Sale Price ($)')
axes[1].set_ylabel('Density')
axes[1].set_title('KDE of Sale Prices', fontweight='bold')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze the target variable: SalePrice
print("SalePrice Statistics:")
print(f"Mean: ${train_df['SalePrice'].mean():,.2f}")
print(f"Median: ${train_df['SalePrice'].median():,.2f}")
print(f"Std Dev: ${train_df['SalePrice'].std():,.2f}")
print(f"Min: ${train_df['SalePrice'].min():,.2f}")
print(f"Max: ${train_df['SalePrice'].max():,.2f}")
print(f"Skewness: {train_df['SalePrice'].skew():.4f}")

## 3. Target Variable Analysis

In [ ]:
# Missing values analysis
missing_values = train_df.isnull().sum()
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_values,
    'Percentage': missing_percent
}).sort_values('Missing_Count', ascending=False)

print("\nMissing Values:")
print(missing_df[missing_df['Missing_Count'] > 0])

# Visualize missing values
plt.figure(figsize=(12, 8))
sns.heatmap(train_df.isnull(), cbar=True, cmap='viridis', yticklabels=False)
plt.title('Missing Value Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.tight_layout()
plt.show()

In [ ]:
# Statistical summary of the dataset
print("Statistical Summary:")
print(train_df.describe())

In [ ]:
# Display dataset shape and basic information
print("Dataset Information:")
print(f"Shape: {train_df.shape}")
print(f"\nData Types:")
print(train_df.dtypes.value_counts())

print(f"\nFirst 10 rows:")
print(train_df.head(10))

## 2. Dataset Overview

In [ ]:
# Load the Kaggle House Prices dataset
loader = DataLoader()
train_df, test_df = loader.load_data('../data/train.csv', '../data/test.csv')

# Display first few rows
print("\nFirst 5 rows of training data:")
print(train_df.head())

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Import our custom data loader
import sys
sys.path.append('..')
from src.data_loader import DataLoader

## 1. Setup — Imports and Load Data

# House Price Prediction - Exploratory Data Analysis

This notebook contains comprehensive exploratory data analysis for the Kaggle House Prices dataset.
We'll investigate dataset structure, target variable distribution, feature correlations, and identify potential outliers.